In [122]:
import numpy as np
import pandas as pd

In [123]:
df = pd.read_csv("../data/data_cleaned-v3.csv")

In [124]:
remove_cols = [
    "spec_Engine Type",
    "spec_Drivetrain",
    "spec_Acceleration",
    "spec_Ground Clearance",
    "spec_Brakes",
    "spec_Front Suspension",
    "spec_Rear Suspension",
    "spec_Tyre Pressure Monitoring System",
    "spec_NCAP Rating",
    "spec_Child Safety Rating",
    "spec_Parking Assist",
    "spec_Audio System",
    "spec_Bluetooth Compatibility",
    "spec_Infotainment Screen",
    "spec_Steering-Mounted Controls",
    "spec_Voice Command",
    "spec_Instrument Cluster Screen",
    "spec_Wireless Charger",
    "spec_Forward Collision Warning",
    "spec_Lane Departure Warning",
    "spec_Blind Spot Detection",
    "spec_High-beam Assist",
    "spec_Traffic Sign Recognition",
    "spec_Electric Motor",
    "spec_Range (As tested by CarWale)",
    "spec_Officially Certified Range (km)",
    "spec_Motor Warranty (In Years)",
    "spec_Motor Warranty (In Kilometres)",
    "spec_Charger Included with Vehicle",
    "spec_Portable EV Charging Cable",
    "spec_Charging Options",
    "spec_Pure Electric Driving Mode",
    "spec_Vehicle to load technology (V2L)",
    "spec_Vehicle to Vehicle charging (V2V)",
    "spec_Fuel Change Over Switch",
    "spec_Direct Start in CNG",
    "spec_Drive Modes",
]

In [125]:
df.drop(columns=remove_cols, inplace=True)

In [126]:
df.shape

(29946, 53)

In [127]:
na_values = df.isna().sum()
na_values[na_values > 20000]

spec_Top Speed                   28488
spec_ADAS                        27493
spec_Driving Range               20019
spec_DC Fast Charging            29727
spec_AC Fast Charging            29727
spec_AC Regular Charging         29764
spec_Regenerative Braking        26642
spec_CNG Tank Capacity (L/Kg)    29767
dtype: int64

In [128]:
df.columns.to_list()

['Price',
 'title',
 'new_car_price',
 'overview_Manufacturing Year',
 'overview_Registration year',
 'overview_Kilometer',
 'overview_No. of owners',
 'overview_Fuel type',
 'overview_Transmission',
 'overview_Color',
 'overview_Insurance',
 'overview_Registration Type',
 'overview_Car Available at',
 'spec_Engine',
 'spec_Fuel Type',
 'spec_Transmission',
 'spec_Max Power (bhp@rpm)',
 'spec_Max Torque (Nm@rpm)',
 'spec_Mileage',
 'spec_Top Speed',
 'spec_Emission Standard',
 'spec_Seating Capacity',
 'spec_Bootspace',
 'spec_Fuel Tank Capacity',
 'spec_Airbags',
 'spec_Anti-Lock Braking System (ABS)',
 'spec_Electronic Stability Program (ESP)',
 'spec_Brake Assist (BA)',
 'spec_Electronic Brake-force Distribution (EBD)',
 'spec_Traction Control System (TC/TCS)',
 'spec_Hill Hold Control',
 'spec_Hill Descent Control',
 'spec_Child Safety Lock',
 'spec_Child Seat Anchor Points',
 'spec_Engine Immobiliser',
 'spec_Central Locking',
 'spec_Sunroof',
 'spec_Parking Sensors',
 'spec_Keyle

## Cleaning the price columns

In [129]:
def clean_price(price: str):
    try:
        price = str(price)
        price = price.replace("Rs.","")
        price = price.replace(",","")
        price = price.strip()
        num_val = "" 
        for i in price:
            if i in "01234567890.":
                num_val += i
        num_val = float(num_val)
        if "Lakh" in price:
            return num_val * 100000
        elif "Crore" in price:
            return num_val * 10000000
        return num_val
    except:
        return np.nan
    
df["Price"] = df.Price.apply(clean_price)

In [130]:
df.Price.isna().sum()

np.int64(933)

## Dropping the rows where Price is missing

In [131]:
df.dropna(subset="Price",inplace=True)

In [132]:
df.Price.isna().sum()

np.int64(0)

In [133]:
df.shape

(29013, 53)

In [134]:
df[["title","overview_Manufacturing Year"]].isna().sum()

title                            0
overview_Manufacturing Year    861
dtype: int64

## Extracting manufacturing year, Brand, Model and Variant from the title

In [135]:
df["Manufacturing Year"] =df.title.str.split(" ").str[0]
df["Brand"] = df.title.str.split(" ").str[1]
df["Model"] = df.title.str.split(" ").str[2]
df["Varient"] = df.title.str.split(" ").str[3:]

In [136]:
df.drop(columns=["title","overview_Manufacturing Year"],inplace=True)

## Cleaning new_car_price column

In [137]:
df["new_car_price"] = df.new_car_price.apply(clean_price)

## Removing the Resgistration Year column as it has so many missing values

In [138]:
df.drop(columns=["overview_Registration year"],inplace=True)

## Cleaning overview_Kilometer columns

In [139]:
def clean_km(km: str):
    try:
        km = km.replace("km","")
        km = km.replace(",","")
        km = km.strip()
        return int(km)
    except:
        np.nan
df["Kilometer"] = df["overview_Kilometer"].apply(clean_km)

In [140]:
df.drop(columns=["overview_Kilometer"],inplace=True)

In [141]:
df["overview_No. of owners"].value_counts()

overview_No. of owners
First               20958
Second               5625
Third                1034
Fourth                108
4 or More              73
Unregistered Car        2
Name: count, dtype: int64

## Cleaning the overview_No. of owners column

In [ ]:
df["ownership"] = df["overview_No. of owners"].map(
    {
        "First": "1",
        "Second": "2",
        "Third": "3",
        "Fourth": "4",
        "4 or More": "4+",
        "Unregistered Car": "4+",
    }
)

In [ ]:
df.drop(columns=["overview_No. of owners"], inplace=True)

In [145]:
df.to_csv("../data/data_cleaned-v4.csv")